In [ ]:
# import uuid
# print(uuid.uuid4())


9eede06d-ef36-4563-a286-fcc6a0954d92


In [5]:
# ============================================================
# TimeSeriesForecaster Agent – FULL COLAB TEST (SINGLE CELL)
# (Fixes OpenAI/httpx proxies issue + robust output handling)
# ============================================================

# 1) Install compatible deps (OpenAI + httpx fix included)
!pip -q install -U "openai>=1.42.0" "httpx==0.28.1"
!pip -q install -U pandas numpy matplotlib seaborn scikit-learn statsmodels prophet langgraph

# IMPORTANT: If you still see the proxies error after this cell,
# do: Runtime -> Restart runtime, then run this cell again.

# 2) Mock AgentGrid imports BEFORE importing agent
import sys, types

app = types.ModuleType("app")
agents = types.ModuleType("agents")
base = types.ModuleType("base")
registry = types.ModuleType("registry")

class BaseAgent:
    def run(self, inputs):
        raise NotImplementedError

class AgentInput:
    def __init__(self, name, type, description=""):
        self.name = name
        self.type = type
        self.description = description

class AgentOutput:
    def __init__(self, name, type, description=""):
        self.name = name
        self.type = type
        self.description = description

def register_agent(agent_id):
    def decorator(cls):
        cls.agent_id = agent_id
        return cls
    return decorator

base.BaseAgent = BaseAgent
base.AgentInput = AgentInput
base.AgentOutput = AgentOutput
registry.register_agent = register_agent

agents.base = base
agents.registry = registry
app.agents = agents

sys.modules["app"] = app
sys.modules["app.agents"] = agents
sys.modules["app.agents.base"] = base
sys.modules["app.agents.registry"] = registry

print("✅ AgentGrid mocked")

# 3) Load OpenAI key from Colab Secrets
import os
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
assert OPENAI_API_KEY, "❌ OPENAI_API_KEY not found in Colab Secrets (🔑)."
print("✅ OpenAI API key loaded")

# 4) Import your agent (upload timeseries_forecaster_agent.py first!)
from timeseries_forecaster_agent import TimeSeriesForecasterAgent
print("✅ Agent imported")

# 5) Generate synthetic time-series + base64 encode CSV
import pandas as pd
import numpy as np
import base64
from io import StringIO, BytesIO
import matplotlib.pyplot as plt

dates = pd.date_range("2023-01-01", periods=200, freq="D")
values = (
    100
    + np.linspace(0, 25, 200)           # trend
    + 12 * np.sin(np.arange(200) / 7)   # weekly seasonality
    + np.random.normal(0, 3, 200)       # noise
)

df = pd.DataFrame({"date": dates, "value": values})

buf = StringIO()
df.to_csv(buf, index=False)
file_base64 = base64.b64encode(buf.getvalue().encode()).decode()
print("✅ Sample data generated")

# 6) Run agent
agent = TimeSeriesForecasterAgent()

result = agent.run({
    "file_data": file_base64,
    "file_format": "csv",
    "date_column": "date",
    "value_column": "value",
    "forecast_horizon": 30,
    "openai_api_key": OPENAI_API_KEY
})

print("✅ Agent execution returned")

# 7) Robust validation & printing (won't crash on partial failures)
errors = result.get("errors", [])
warnings = result.get("warnings", [])
model_type = (result.get("model_info") or {}).get("type")
mape = (result.get("validation_metrics") or {}).get("mape")
forecast_rows = len((result.get("forecast_data") or {}).get("values", []))
viz = result.get("visualizations") or {}

print("\n--- SUMMARY ---")
print("📊 Model type:", model_type)
print("📈 MAPE:", mape)
print("🔮 Forecast rows:", forecast_rows)
print("⚠️ Warnings:", warnings)
print("❌ Errors:", errors)

# If errors exist, show them clearly and stop before plotting
if errors:
    raise RuntimeError("Agent failed. First error:\n" + str(errors[0]))

# Sanity checks
assert forecast_rows == 30, f"Expected 30 forecast rows, got {forecast_rows}"
assert "forecast" in viz, "Missing forecast visualization"

print("🎉 TEST PASSED – Agent ran successfully")

# 8) Render forecast plot from base64
img_b64 = viz["forecast"].split(",")[1]
img = plt.imread(BytesIO(base64.b64decode(img_b64)))

plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis("off")
plt.title("TimeSeriesForecaster Output")
plt.show()


✅ AgentGrid mocked
✅ OpenAI API key loaded
✅ Agent imported
✅ Sample data generated
✅ Agent execution returned

--- SUMMARY ---
📊 Model type: SARIMA
📈 MAPE: 10.759860928209978
🔮 Forecast rows: 30
⚠️ Warnings: []
❌ Errors: []
🎉 TEST PASSED – Agent ran successfully
